In [ ]:
from pyspark import SparkContext


# Function to convert CSV row into tuple
def parse_employee(row):
    fields = row.split(",")

    return (
        int(fields[0]),      # Employee ID
        fields[1],           # Name
        fields[2],           # Department
        int(fields[3])       # Salary
    )


# Main Program
if __name__ == "__main__":

    # Create Spark Context
    sc = SparkContext("local[*]", "EmployeeAnalysis")

    # Read CSV file
    rdd = sc.textFile("employees.csv")

    # Remove header
    header = rdd.first()
    employee_rdd = rdd.filter(lambda row: row != header)

    # Convert each row into tuple
    employees = employee_rdd.map(parse_employee)



    print("\n===== EMPLOYEES SORTED BY SALARY =====\n")

    sorted_employees = employees.sortBy(
        lambda emp: emp[3],
        ascending=False
    )

    for employee in sorted_employees.collect():
        print(employee)


    print("\n===== DEPARTMENT WISE SALARY TOTAL =====\n")

    department_salary = employees.map(
        lambda emp: (emp[2], emp[3])
    )

    department_totals = department_salary.reduceByKey(
        lambda x, y: x + y
    )

    for department, total in department_totals.collect():
        print(f"{department}: {total}")


    print("\n===== TOP 3 HIGHEST PAID EMPLOYEES =====\n")

    top_three = sorted_employees.take(3)

    for employee in top_three:
        print(employee)

    # Save top 3 employees to file
    sc.parallelize(top_three).saveAsTextFile("top_3_employees")

    print("\nTop 3 employees saved successfully!")

    # Stop Spark Context
    sc.stop()